# DHIS2 Climate Tools – Malaria Gridding & Population Weighting

This notebook demonstrates how DHIS2 Climate Tools can be used to
transform **facility-based malaria data** into **population-weighted
gridded risk surfaces** for Malawi.

The workflow reflects the use case where malaria cases are recorded at
health facilities, while transmission occurs within facility catchment
areas.

------------------------------------------------------------------------

## 1. Objective

-   Pull malaria case data directly from **DHIS2** using Climate Tools
-   Interpolate facility-level cases into a **continuous spatial grid**
-   Mask non-inhabited areas (e.g. **water bodies**)
-   Overlay **population distribution** to generate population-weighted
    malaria surfaces
-   Produce outputs suitable for climate–health analysis and EWARS

------------------------------------------------------------------------

## 2. Import Required Libraries

``` python
from io import StringIO
import pandas as pd
import geopandas as gpd
import rioxarray as rxr

from plot import plotData
from rasterise import rasterize_population
from masking import mask
from gridding import linear_grid
from preparedata import prepare_data
```

------------------------------------------------------------------------

## 3. Pull Malaria Case Data from DHIS2

Facility-level malaria cases are retrieved directly from DHIS2 by
specifying the instance URL, reporting period, data dimension, and
organisation unit level.

``` python
data = prepare_data(
    base_url="url",
    username="username",
    password="pwd",
    dx="jPEcKbn7jmh",
    pe="202501",
    ou_level="4"
)
```

Convert the returned CSV string into a pandas DataFrame:

``` python
dataValues = pd.read_csv(StringIO(data))
```

------------------------------------------------------------------------

## 4. Spatial Gridding of Malaria Cases

Facility point data are interpolated into a continuous spatial surface
using **linear interpolation**.

``` python
lin = linear_grid(dataValues)
```

This gridded surface better represents malaria risk across facility
catchment areas rather than at single points.

------------------------------------------------------------------------

## 5. Masking Non-Populated Areas

Water bodies and other non-inhabited areas are removed from the
interpolated surface to avoid assigning cases where people do not live.

``` python
grd = mask(lin)
```

------------------------------------------------------------------------

## 6. Load Population and Boundary Data

Population distribution data are used to weight malaria cases according
to where people live.

``` python
pop = gpd.read_file(r"C:\\Users\\ShnkMn\\Documents\\CMS\\climate-tools\\docs\\data\\pop.gpkg")
pop = pop.to_crs(epsg=4326)
```

Administrative boundaries are used for visualization and aggregation.

``` python
overlay = gpd.read_file(r"C:\\Users\\ShnkMn\\Documents\\CMS\\climate-tools\\docs\\data\\Districts.shp")
overlay = overlay.to_crs(epsg=4326)
```

------------------------------------------------------------------------

## 7. Rasterize Population Distribution

Convert population polygons into a raster aligned with the malaria grid.

``` python
rst = rasterize_population(pop, lin, pop_col="population")
```

Ensure spatial alignment with the malaria grid:

``` python
rst = rst.reindex_like(lin, method=None)
```

------------------------------------------------------------------------

## 8. Generate Population Weights

Mask population raster to valid malaria grid cells:

``` python
spatial_mask = lin.isel(time=0).notnull()
rst_masked = rst.where(spatial_mask)
```

Calculate population totals and weights:

``` python
pop_total = rst_masked.sum(dim=("lat", "lon"))
weights = rst_masked / pop_total
```

------------------------------------------------------------------------

## 9. Redistribute Malaria Cases by Population

Calculate total malaria cases from the interpolated grid:

``` python
total_cases = lin.isel(time=0).sum(dim=("lat", "lon"))
```

Apply population weights to redistribute cases:

``` python
cases = weights * total_cases
```

Mask non-inhabited areas again:

``` python
msk = mask(cases)
print(msk)
```

------------------------------------------------------------------------

## 10. Visualization

Plot the interpolated malaria surface:

``` python
plotData(grd, overlay)
```

Plot the population-weighted malaria surface:

``` python
plotData(msk, overlay)
```

------------------------------------------------------------------------

## 11. Outputs and Use Cases

**Outputs**: - Gridded malaria risk surfaces - Population-adjusted
malaria burden maps

**Use Cases**: - Climate–malaria modelling - Early Warning and Response
Systems (EWARS) - Targeted malaria interventions at sub-district level

------------------------------------------------------------------------

